# Week 0 — Setup check

Run each cell from top to bottom (**Runtime → Run all** also works). This notebook:

1. installs the course helpers,
2. mounts your Google Drive, so the course dataset survives a Colab reset,
3. generates the course dataset (a fixed seed, so everyone gets identical data),
4. calls Claude once with the key from **Colab Secrets**, and
5. runs the setup self-check.

Before you start: in the left sidebar, click the **key icon (Secrets)** → **Add new secret** → name `ANTHROPIC_API_KEY` → paste your key → turn on **Notebook access**. Never paste the key into a cell.

In [ ]:
# Setup (run first): installs the course helpers, mounts Google Drive, prepares the dataset.
import subprocess, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "course-tools @ git+https://github.com/YOUR-GITHUB-ORG/harness-starter@v1#subdirectory=course-tools"],
                   check=True)
    from google.colab import drive
    drive.mount("/content/drive")   # approve the pop-up; the dataset is saved to MyDrive/harness-course

import course_tools as ct
env = ct.bootstrap()

## Your first call to Claude

Before you run the next cell, predict: what type is `response`, and where in it will the reply text be?

In [ ]:
import anthropic

client = anthropic.Anthropic()          # reads ANTHROPIC_API_KEY from the environment; never type it here
response = client.messages.create(
    model=ct.MODELS["haiku"],
    max_tokens=60,
    messages=[{"role": "user", "content": "In one sentence, what does a goods receipt note record?"}],
)
print(type(response))
print(response.content[0].text)
print(response.usage)

## Versions

Everyone in the cohort runs the same library versions. If something behaves differently for you, compare these first.

In [ ]:
import importlib.metadata as md, platform
print("Python", platform.python_version())
for pkg in ["anthropic", "langchain", "langchain-anthropic", "langgraph", "pydantic", "pymupdf", "course-tools"]:
    print(f"{pkg:20} {md.version(pkg)}")

## A first look at the dataset

The dataset belongs to a fictional FMCG distributor, **Nilgiri Distributors**, in Bengaluru. Everything is synthetic.

In [ ]:
import sqlite3
import pandas as pd

con = sqlite3.connect(env.db)
for table in ["suppliers", "skus", "purchase_orders", "po_lines", "goods_receipts", "grn_lines", "invoices"]:
    n = con.execute(f"select count(*) from {table}").fetchone()[0]
    print(f"{table:16} {n:5} rows")
pd.read_csv(env.data / "catalogue.csv").head(10)

## Self-check

The last line should read `Secrets OK · Claude reachable · Drive mounted · Dataset OK`.

Then save this notebook into your own repository: **File → Save a copy in GitHub** → choose your repository → set the path to `week-00/setup.ipynb`. When GitHub asks for authorisation, tick the option to include private repositories. Save again after every edit: Colab does not sync on its own.

In [ ]:
ct.selfcheck(0)